# Piloto one-class — anomalib 2.x (PatchCore / PaDiM)

**Pré-requisitos:** rode as células **em ordem, de cima para baixo**. A célula 2 (imports+dataset) define `Patchcore`, `Padim`, `Engine` e `dm`; sem ela, a célula de treino avisa em vez de dar `NameError`. API 2.x: datamodule `MVTecAD`; **não há `CutPaste`**; **não usa `imgaug`**.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q anomalib

## 1) Imports + dataset (rode primeiro — define Patchcore/Padim/Engine/dm)

In [ ]:
import os, pathlib
from anomalib.data import MVTecAD
from anomalib.engine import Engine
from anomalib.models import Patchcore, Padim, Fastflow

# ROOT dinâmico: Colab usa /content; local (<host>) usa a pasta ja baixada
ROOT = "/content/datasets/MVTecAD"
if not os.path.isdir("/content"):
    ROOT = str(pathlib.Path.home() / "tcc-pnaat/datasets/MVTecAD")
pathlib.Path(ROOT).mkdir(parents=True, exist_ok=True)

dm = MVTecAD(root=ROOT, category="bottle", train_batch_size=32, eval_batch_size=32)
dm.prepare_data()   # baixa MVTecAD (~5GB) na 1a vez; local ja existe -> ignora
dm.setup()
print("ok: MVTecAD bottle em", ROOT)

## 2) Exemplos do dataset (preview)

In [ ]:
import glob
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

def grid(paths, title, n=4):
    paths = paths[:n]
    if not paths: print("(sem imagens:", title, ")"); return
    fig, axes = plt.subplots(1, len(paths), figsize=(3*len(paths), 3))
    if len(paths) == 1: axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(Image.open(p)); ax.set_title(Path(p).parent.name, fontsize=9); ax.axis("off")
    fig.suptitle(title); plt.tight_layout(); plt.show()

base = Path(ROOT)/"bottle"
print("categorias:", [p.name for p in base.iterdir()] if base.exists() else "NAO ENCONTRADO")
grid(sorted(glob.glob(str(base/'train'/'good'/'*.png'))), "TREINO (good)")
for d in sorted(glob.glob(str(base/'test'/'*'))):
    grid(sorted(glob.glob(d+'/*.png')), "TESTE: " + Path(d).name)

## 3) Treino + avaliação (PatchCore e PaDiM)

Rode esta célula **depois** da célula 1 (imports+dataset). Contém guarda: se os pré-requisitos não rodaram, avisa.

In [ ]:
if "Patchcore" not in globals() or "dm" not in globals():
    raise RuntimeError("Rode primeiro a célula '1) Imports + dataset'.")

def run(model, epochs, nome):
    eng = Engine(max_epochs=epochs, accelerator="auto", devices=1, enable_progress_bar=False)
    eng.fit(model=model, datamodule=dm)
    res = eng.test(model=model, datamodule=dm)
    m = res[0] if isinstance(res, (list, tuple)) and res else res
    print("===", nome, "==="); print(m)
    return nome, m

res = []
res.append(run(Patchcore(), 1, "PatchCore"))
res.append(run(Padim(), 1, "PaDiM"))

## 4) (Opcional) FastFlow — mais épocas

In [ ]:
res.append(run(Fastflow(), 20, "FastFlow"))

## 5) (Opcional) EfficientAd — candidato de borda

In [ ]:
try:
    from anomalib.models import EfficientAd
    res.append(run(EfficientAd(), 10, "EfficientAd"))
except Exception as e:
    print("EfficientAd indisponivel:", e)

## 6) Comparativo

In [ ]:
import pandas as pd
rows = [{"modelo": nome, **{k:v for k,v in m.items() if k.lower().endswith(("auroc","aupro","f1"))}}
        for nome, m in res if isinstance(m, dict)]
print(pd.DataFrame(rows).to_string(index=False))

## 7) (Opcional) Pontuar a NOSSA imagem

Copie a foto para `ROOT/bottle/test/<tipo>/` (ou `good`) e use `anomalib predict`. Lista checkpoints:

In [ ]:
import glob
print(glob.glob("/content/results/**/*.ckpt", recursive=True)[:5])